# Advanced Problems with Solutions: Decimal Math Operations

Topic focus: `Decimal` division, modulo, `divmod()`, precision, context, and Decimal mathematical functions.

In [1]:
import decimal
from decimal import Decimal, getcontext, localcontext
import math

## Problem 1: Integer vs Decimal Division with Negative Numbers

For each pair `(n, d)`, compute `n // d`, `n % d`, and `divmod(n, d)` using both `int` and `Decimal` values.

Pairs:

```python
(-17, 5)
(17, -5)
(-17, -5)
```

Explain why the results differ.

In [2]:
pairs = [(-17, 5), (17, -5), (-17, -5)]

for n, d in pairs:
    print(f'int:     n={n}, d={d}')
    print('  //:', n // d)
    print('  % :', n % d)
    print('  divmod:', divmod(n, d))
    print('  identity:', n == d * (n // d) + (n % d))

    N = Decimal(n)
    D = Decimal(d)
    print(f'Decimal: n={N}, d={D}')
    print('  //:', N // D)
    print('  % :', N % D)
    print('  divmod:', divmod(N, D))
    print('  identity:', N == D * (N // D) + (N % D))
    print('-' * 40)

int:     n=-17, d=5
  //: -4
  % : 3
  divmod: (-4, 3)
  identity: True
Decimal: n=-17, d=5
  //: -3
  % : -2
  divmod: (Decimal('-3'), Decimal('-2'))
  identity: True
----------------------------------------
int:     n=17, d=-5
  //: -4
  % : -3
  divmod: (-4, -3)
  identity: True
Decimal: n=17, d=-5
  //: -3
  % : 2
  divmod: (Decimal('-3'), Decimal('2'))
  identity: True
----------------------------------------
int:     n=-17, d=-5
  //: 3
  % : -2
  divmod: (3, -2)
  identity: True
Decimal: n=-17, d=-5
  //: 3
  % : -2
  divmod: (Decimal('3'), Decimal('-2'))
  identity: True
----------------------------------------


### Solution

For integers, `//` performs floor division. The quotient is rounded toward negative infinity.

For `Decimal`, `//` performs truncated division. The quotient is rounded toward zero.

That is why negative cases may differ.

However, both systems preserve the identity:

```python
n == d * (n // d) + (n % d)
```

## Problem 2: Predict the Decimal Remainder Sign

Without running the code first, predict the sign of the remainder for each expression:

```python
Decimal('-25') % Decimal('4')
Decimal('25') % Decimal('-4')
Decimal('-25') % Decimal('-4')
```

Then verify your answer.

In [3]:
cases = [
    (Decimal('-25'), Decimal('4')),
    (Decimal('25'), Decimal('-4')),
    (Decimal('-25'), Decimal('-4')),
]

for n, d in cases:
    q = n // d
    r = n % d
    print(f'{n} // {d} = {q}')
    print(f'{n} %  {d} = {r}')
    print('identity:', n == d * q + r)
    print()

-25 // 4 = -6
-25 %  4 = -1
identity: True

25 // -4 = -6
25 %  -4 = 1
identity: True

-25 // -4 = 6
-25 %  -4 = -1
identity: True



### Solution

With `Decimal`, the quotient is truncated toward zero.

So the remainder follows this identity:

```python
n == d * q + r
```

For `Decimal`, the remainder has the same sign as the dividend `n`, not necessarily the divisor `d`.

Expected results:

```python
Decimal('-25') % Decimal('4')   == Decimal('-1')
Decimal('25') % Decimal('-4')   == Decimal('1')
Decimal('-25') % Decimal('-4')  == Decimal('-1')
```

## Problem 3: Implement Decimal-Style Division for Integers

Write a function `decimal_style_divmod(n, d)` that accepts integers but returns the same quotient and remainder behavior as `Decimal`.

It should truncate the quotient toward zero instead of flooring it.

In [4]:
def decimal_style_divmod(n, d):
    if d == 0:
        raise ZeroDivisionError('division by zero')
    q = abs(n) // abs(d)
    if (n < 0) != (d < 0):
        q = -q
    r = n - d * q
    return q, r


tests = [(-17, 5), (17, -5), (-17, -5), (17, 5)]

for n, d in tests:
    custom = decimal_style_divmod(n, d)
    dec = divmod(Decimal(n), Decimal(d))
    print((n, d), custom, dec, custom == tuple(map(int, dec)))

(-17, 5) (-3, -2) (Decimal('-3'), Decimal('-2')) True
(17, -5) (-3, 2) (Decimal('-3'), Decimal('2')) True
(-17, -5) (3, -2) (Decimal('3'), Decimal('-2')) True
(17, 5) (3, 2) (Decimal('3'), Decimal('2')) True


### Solution

The key is to compute the quotient using absolute values first, then apply the sign manually.

`Decimal` division truncates toward zero, so:

```python
Decimal('-17') // Decimal('5') == Decimal('-3')
```

whereas integer floor division gives:

```python
-17 // 5 == -4
```

The remainder is then computed from the invariant:

```python
r = n - d * q
```

## Problem 4: Detect Accidental Float Conversion

A developer writes this function:

```python
def bad_square_root(x):
    return Decimal(math.sqrt(x))
```

Explain why this is dangerous when `x` is a `Decimal`.

Then write a safer version.

In [5]:
def bad_square_root(x):
    return Decimal(math.sqrt(x))


def good_square_root(x):
    if not isinstance(x, Decimal):
        x = Decimal(str(x))
    return x.sqrt()


x = Decimal('0.01')

print('bad :', bad_square_root(x))
print('good:', good_square_root(x))
print('bad squared :', bad_square_root(x) * bad_square_root(x))
print('good squared:', good_square_root(x) * good_square_root(x))

bad : 0.1000000000000000055511151231257827021181583404541015625
good: 0.1
bad squared : 0.01000000000000000111022302463
good squared: 0.01


### Solution

`math.sqrt()` converts the input to a binary floating-point number.

That means the exact decimal value may be lost before the result is converted back into a `Decimal`.

The safer version uses the `Decimal.sqrt()` method directly:

```python
x.sqrt()
```

## Problem 5: Precision and Context

Compute the square root of `2` using Decimal precision values of `10`, `20`, and `50`.

Then square each result and compare it with `2`.

In [6]:
for precision in [10, 20, 50]:
    with localcontext() as ctx:
        ctx.prec = precision
        x = Decimal('2')
        root = x.sqrt()
        squared = root * root
        print(f'precision = {precision}')
        print('sqrt(2)   =', root)
        print('squared   =', squared)
        print('error     =', squared - x)
        print('-' * 50)

precision = 10
sqrt(2)   = 1.414213562
squared   = 1.999999999
error     = -1E-9
--------------------------------------------------
precision = 20
sqrt(2)   = 1.4142135623730950488
squared   = 2.0000000000000000000
error     = 0E-19
--------------------------------------------------
precision = 50
sqrt(2)   = 1.4142135623730950488016887242096980785696718753769
squared   = 1.9999999999999999999999999999999999999999999999999
error     = -1E-49
--------------------------------------------------


### Solution

`Decimal` operations are controlled by the current decimal context.

Increasing `ctx.prec` gives more significant digits.

However, irrational results such as `sqrt(2)` cannot be represented exactly, so squaring the rounded result may not return exactly `2`.

## Problem 6: Build a Decimal-Safe Hypotenuse Function

Write a function `decimal_hypot(a, b, precision=28)` that computes:

```python
sqrt(a*a + b*b)
```

using only `Decimal` arithmetic.

The function should accept strings, integers, or Decimals.

In [7]:
def to_decimal(value):
    if isinstance(value, Decimal):
        return value
    return Decimal(str(value))


def decimal_hypot(a, b, precision=28):
    with localcontext() as ctx:
        ctx.prec = precision
        a = to_decimal(a)
        b = to_decimal(b)
        return (a * a + b * b).sqrt()


print(decimal_hypot('3', '4'))
print(decimal_hypot('0.1', '0.2', precision=50))
print(decimal_hypot(Decimal('1.23456789'), Decimal('9.87654321'), precision=40))

5
0.22360679774997896964091736687312762354406183596115
9.953404626258100444872509303348631533171


### Solution

The function avoids `math.hypot()` because `math.hypot()` uses binary floating-point arithmetic.

The helper function converts inputs through `str(value)` instead of directly using `Decimal(value)`, which helps avoid importing binary float artifacts when possible.

## Problem 7: Decimal Logarithms vs Float Logarithms

Compare the following two approaches:

```python
Decimal('1.01').ln()
Decimal(math.log(1.01))
```

Show the difference between the two results.

In [8]:
with localcontext() as ctx:
    ctx.prec = 50

    x_dec = Decimal('1.01')
    decimal_result = x_dec.ln()

    float_result = Decimal(math.log(1.01))

    print('Decimal ln :', decimal_result)
    print('Float ln   :', float_result)
    print('Difference :', decimal_result - float_result)

Decimal ln : 0.0099503308531680828482153575442607416886796099400588
Float ln   : 0.009950330853168092015703649622082593850791454315185546875
Difference : -9.167488292077821852162111844375126746875E-18


### Solution

`Decimal('1.01').ln()` keeps the computation in Decimal arithmetic.

`math.log(1.01)` first uses a binary float approximation of `1.01`, then returns a float.

Converting that float back to `Decimal` does not recover the lost precision.

## Problem 8: Financial Remainder Allocation

A payment of `$100.00` must be split equally among `6` people.

Use `Decimal` to compute:

1. The base amount each person receives to the nearest cent, rounded down.
2. The leftover amount.
3. A final allocation list that distributes the leftover cents fairly.

In [9]:
total = Decimal('100.00')
people = 6
cent = Decimal('0.01')

raw_share = total / people
base_share = raw_share.quantize(cent, rounding=decimal.ROUND_DOWN)
allocated = base_share * people
leftover = total - allocated

extra_cents = int(leftover / cent)
shares = [base_share] * people

for i in range(extra_cents):
    shares[i] += cent

print('raw_share :', raw_share)
print('base_share:', base_share)
print('leftover  :', leftover)
print('shares    :', shares)
print('sum       :', sum(shares))

raw_share : 16.66666666666666666666666667
base_share: 16.66
leftover  : 0.04
shares    : [Decimal('16.67'), Decimal('16.67'), Decimal('16.67'), Decimal('16.67'), Decimal('16.66'), Decimal('16.66')]
sum       : 100.00


### Solution

The exact split is repeating:

```python
100 / 6 = 16.6666...
```

For money, each individual amount must usually be rounded to cents.

Rounding each share down gives `$16.66`, leaving `$0.04`.

The fair allocation is four people receiving `$16.67` and two people receiving `$16.66`.

## Problem 9: Verify Decimal Exponential and Logarithm Inverses

For each value below, compute `x.ln().exp()` and compare it with `x`:

```python
Decimal('0.5')
Decimal('1.25')
Decimal('10')
```

Use precision `30`.

In [10]:
values = [Decimal('0.5'), Decimal('1.25'), Decimal('10')]

with localcontext() as ctx:
    ctx.prec = 30

    for x in values:
        result = x.ln().exp()
        print('x       :', x)
        print('result  :', result)
        print('error   :', result - x)
        print('-' * 40)

x       : 0.5
result  : 0.500000000000000000000000000000
error   : 0E-30
----------------------------------------
x       : 1.25
result  : 1.25000000000000000000000000000
error   : 0E-29
----------------------------------------
x       : 10
result  : 9.99999999999999999999999999996
error   : -4E-29
----------------------------------------


### Solution

Mathematically:

```python
exp(ln(x)) == x
```

But Decimal computations are rounded according to the active context precision.

So the result should be extremely close to `x`, though not necessarily exactly equal for every input and precision.

## Problem 10: Write a Robust Decimal Calculator Function

Write a function `decimal_calculate(a, b, op, precision=28)` supporting:

- `'add'`
- `'sub'`
- `'mul'`
- `'div'`
- `'floordiv'`
- `'mod'`
- `'divmod'`
- `'sqrt_a'`
- `'ln_a'`
- `'exp_a'`

The function should use Decimal arithmetic and should not use the `math` module.

In [11]:
def decimal_calculate(a, b=None, op='add', precision=28):
    with localcontext() as ctx:
        ctx.prec = precision

        a = to_decimal(a)
        b = None if b is None else to_decimal(b)

        if op == 'add':
            return a + b
        if op == 'sub':
            return a - b
        if op == 'mul':
            return a * b
        if op == 'div':
            return a / b
        if op == 'floordiv':
            return a // b
        if op == 'mod':
            return a % b
        if op == 'divmod':
            return divmod(a, b)
        if op == 'sqrt_a':
            return a.sqrt()
        if op == 'ln_a':
            return a.ln()
        if op == 'exp_a':
            return a.exp()

        raise ValueError(f'Unsupported operation: {op}')


print(decimal_calculate('-10', '3', 'floordiv'))
print(decimal_calculate('-10', '3', 'mod'))
print(decimal_calculate('2', op='sqrt_a', precision=50))
print(decimal_calculate('1.5', op='ln_a', precision=40))
print(decimal_calculate('1.5', op='exp_a', precision=40))

-3
-1
1.4142135623730950488016887242096980785696718753769
0.4054651081081643819780131154643491365720
4.481689070338064822602055460119275819006


### Solution

A robust Decimal calculator should:

- Convert inputs safely using `Decimal(str(value))`.
- Use `localcontext()` so precision changes do not leak globally.
- Use Decimal methods like `.sqrt()`, `.ln()`, and `.exp()`.
- Avoid `math.sqrt`, `math.log`, and `math.exp` when precision matters.
- Preserve Decimal division and modulo behavior.